In [1]:
!pip install scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.7/331.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 4.6 MB/s eta 0:00:00


In [2]:
import os
import scrapy
from scrapy.crawler import CrawlerProcess

In [3]:
if not os.path.exists('ria_parser'):
    !scrapy startproject ria_parser
    print("Проект создан")
else:
    print("Проект уже существует")

New Scrapy project 'ria_parser', using template directory '/usr/local/lib/python3.12/dist-packages/scrapy/templates/project', created in:
    /content/ria_parser

You can start your first spider with:
    cd ria_parser
    scrapy genspider example example.com
Проект создан


In [4]:
%cd ria_parser

/content/ria_parser


In [5]:
!scrapy genspider ria_news ria.ru

Created spider 'ria_news' using template 'basic' in module:
  ria_parser.spiders.ria_news


In [6]:
print("Паук создан")

Паук создан


In [7]:
%%writefile ria_parser/spiders/ria_news.py
import scrapy
from datetime import datetime

class RiaNewsSpider(scrapy.Spider):
    name = "ria_news"
    allowed_domains = ["ria.ru"]
    start_urls = ["https://ria.ru/"]

    custom_settings = {
        'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'DOWNLOAD_DELAY': 1.5,
        'CONCURRENT_REQUESTS_PER_DOMAIN': 1,
        'ROBOTSTXT_OBEY': False,
        'COOKIES_ENABLED': False,
        'RETRY_ENABLED': True,
        'RETRY_TIMES': 3,
    }

    def parse(self, response):
        """Парсим главную страницу ria.ru"""
        self.logger.info(f"Парсим страницу: {response.url}")

        news_items = response.css('a.list-item__title')

        if not news_items:

            news_items = response.css('a[href*="/"]:has(strong), h3 a, .cell-list__item a')

        self.logger.info(f"Найдено элементов: {len(news_items)}")

        for item in news_items:
            title = item.css('::text').get()
            link = item.css('::attr(href)').get()

            if title and link and len(title.strip()) > 10:
                yield {
                    'title': title.strip(),
                    'url': response.urljoin(link),
                    'timestamp': datetime.now().isoformat(),
                }

Overwriting ria_parser/spiders/ria_news.py


In [8]:
%%writefile ria_parser/settings.py

BOT_NAME = "ria_parser"

SPIDER_MODULES = ["ria_parser.spiders"]
NEWSPIDER_MODULE = "ria_parser.spiders"

ROBOTSTXT_OBEY = False

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

DOWNLOAD_DELAY = 2.0
CONCURRENT_REQUESTS_PER_DOMAIN = 1
CONCURRENT_REQUESTS = 8

AUTOTHROTTLE_ENABLED = True
AUTOTHROTTLE_START_DELAY = 1.0
AUTOTHROTTLE_MAX_DELAY = 30.0
AUTOTHROTTLE_TARGET_CONCURRENCY = 1.0

RETRY_ENABLED = True
RETRY_TIMES = 5
RETRY_HTTP_CODES = [500, 502, 503, 504, 403, 404, 408]

COOKIES_ENABLED = False

HTTPCACHE_ENABLED = True
HTTPCACHE_EXPIRATION_SECS = 300
HTTPCACHE_DIR = "httpcache"

FEED_EXPORT_ENCODING = 'utf-8'

Overwriting ria_parser/settings.py


In [9]:
%cd /content/ria_parser

!scrapy crawl ria_news -o ria_news_results.json

print("Паук выполнился. Результат сохранён в ria_news_results.json")

/content/ria_parser
2026-03-10 13:31:11 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: ria_parser)
2026-03-10 13:31:11 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
2026-03-10 13:31:11 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-03-10 13:31:11 [asyncio] DEBUG: Using selector: EpollSelector
2026-03-10 13:31:11 [scrapy.addons] INFO: Enabled addons:
[]
2026-03-10 13:31:11 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-03-10 13:31:11 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-03-10 13:31:11 [scrapy.extensions.telnet] INFO: Telnet Password: 0f23b317a177e95f
202

In [11]:
import json
import pandas as pd

In [12]:
try:
    with open('ria_news_results.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"Всего собрано новостей: {len(data)}")

    df = pd.DataFrame(data)
    print("\nПервые 10 записей:")
    print(df[['title']].head(10))

    df.to_csv('ria_news_results.csv', index=False)
    print("\nРезультаты также сохранены в ria_news_results.csv")

except FileNotFoundError:
    print("Файл с результатами не найден. Проверь, выполнился ли паук.")
except json.JSONDecodeError:
    print("Файл пуст или повреждён. Возможно, паук ничего не нашёл.")

Всего собрано новостей: 27

Первые 10 записей:
                                               title
0  Песков раскрыл подробности переговоров президе...
1  ВОЗ получила сообщения об использовании Израил...
2  В Кремле объяснили возможные ограничения связи...
3  Средняя цена на газ в Европе достигла максимум...
4  Россия завоевала второе золото на Паралимпиаде...
5        В США сообщили о начавшейся у Трампа панике
6  "Публичный позор". Дерзкая выходка Зеленского ...
7    Иран заявил об атаке на израильский НПЗ в Хайфе
8  Правительство отчитается перед ЕР о выполнении...
9  Израиль нанес удар по школе в иранской провинц...

Результаты также сохранены в ria_news_results.csv


In [19]:
import json
import os
from datetime import datetime

def update_news_data(new_file='ria_news_results.json', archive_file='ria_news_archive.json'):
    """
    Добавляет только новые новости в архив.
    Читает JSON Lines формат Scrapy (каждая строка - отдельный JSON)
    """
    new_news = []
    try:
        with open(new_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # пропускаем пустые строки
                    try:
                        item = json.loads(line)
                        new_news.append(item)
                    except json.JSONDecodeError as e:
                        print(f"Ошибка в строке: {line[:50]}... - {e}")
                        continue
        print(f"Загружено новых записей из файла: {len(new_news)}")
    except FileNotFoundError:
        print(f"Файл {new_file} не найден")
        return 0

    archive = []
    if os.path.exists(archive_file):
        try:
            with open(archive_file, 'r', encoding='utf-8') as f:
                archive = json.load(f)
            print(f"Загружен архив: {len(archive)} записей")
        except json.JSONDecodeError:
            print("Архив повреждён, создаём новый")
            archive = []

    # Создаём множество существующих ключей для быстрой проверки
    existing_keys = {(item.get('title', ''), item.get('url', '')) for item in archive}

    added = 0
    for news in new_news:
        # Проверяем, что у записи есть нужные поля
        if 'title' not in news or 'url' not in news:
            continue

        key = (news['title'], news['url'])
        if key not in existing_keys:
            # Добавляем дату добавления в архив
            news['added_to_archive'] = datetime.now().isoformat()
            archive.append(news)
            existing_keys.add(key)
            added += 1

    # Сохраняем обновлённый архив
    with open(archive_file, 'w', encoding='utf-8') as f:
        json.dump(archive, f, ensure_ascii=False, indent=2)

    print(f"Добавлено новых новостей: {added}")
    print(f"Всего в архиве: {len(archive)}")

    return added

In [20]:
# Ещё раз запускаем паука
!scrapy crawl ria_news -o ria_news_results.json

added = update_news_data()

2026-03-10 13:43:37 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: ria_parser)
2026-03-10 13:43:37 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
2026-03-10 13:43:37 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-03-10 13:43:37 [asyncio] DEBUG: Using selector: EpollSelector
2026-03-10 13:43:37 [scrapy.addons] INFO: Enabled addons:
[]
2026-03-10 13:43:37 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-03-10 13:43:37 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-03-10 13:43:37 [scrapy.extensions.telnet] INFO: Telnet Password: aa39122f8cfdf51a
2026-03-10 13:43:37 [sc